In [1]:
import pandas as pd 
import numpy as np
from matplotlib import pyplot as plt 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import utils

# SKLearn related imports
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import TransformerMixin, BaseEstimator

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from lightgbm import LGBMRegressor  # or any other estimator

plt.rcParams['figure.figsize'] = (12, 4.2)

In [2]:
### dataset 1
df1 = pd.read_csv("train/chain_campaigns.csv")
# Convert dates and compute duration
df1['start_date'] = pd.to_datetime(df1['start_date'])
df1['end_date'] = pd.to_datetime(df1['end_date'])
df1['duration_days'] = (df1['end_date'] - df1['start_date']).dt.days + 1
df1.head()

,competitor,start_date,end_date,chain_campaign,duration_days
0,chain,2024-08-12,2024-08-18,C2,7
1,competitorA,2023-09-22,2023-09-25,A2,4
2,chain,2024-09-23,2024-09-29,C1,7
3,chain,2024-04-08,2024-04-14,C1,7
4,competitorA,2023-10-20,2023-10-23,A2,4


In [3]:
### dataset 2
df2 = pd.read_csv("train/product_prices_leaflets.csv")
df2['time_key'] = pd.to_datetime(df2['time_key'].astype(str), format='%Y%m%d') #convert time_key to datetime
#df2 = df2.set_index(['sku','time_key']).sort_index()

#drop negative discount values
neg_count = (df2['discount'] < 0).sum()
print(f"Dropping {neg_count} rows with negative discount")

# Filter them out
df2 = df2[df2['discount'] >= 0].copy()

df2['effective_price'] = df2['pvp_was'] * (1 - df2['discount'])

df2.head()

Dropping 40 rows with negative discount


,sku,time_key,competitor,pvp_was,discount,flag_promo,leaflet,effective_price
0,2056,2024-03-12,chain,21.70,0.0000,0,NaN,21.700000
1,4435,2023-06-21,chain,18.92,0.2754,1,weekly,13.709432
2,1951,2023-03-03,competitorA,60.58,0.3524,1,NaN,39.231608
3,2135,2024-06-05,chain,55.37,0.2509,1,weekly,41.477667
4,3949,2023-05-29,chain,51.90,0.0000,0,NaN,51.900000


In [4]:
df2['leaflet'] = df2['leaflet'].fillna('no promotion')

In [5]:
df2.head()

,sku,time_key,competitor,pvp_was,discount,flag_promo,leaflet,effective_price
0,2056,2024-03-12,chain,21.70,0.0000,0,no promotion,21.700000
1,4435,2023-06-21,chain,18.92,0.2754,1,weekly,13.709432
2,1951,2023-03-03,competitorA,60.58,0.3524,1,no promotion,39.231608
3,2135,2024-06-05,chain,55.37,0.2509,1,weekly,41.477667
4,3949,2023-05-29,chain,51.90,0.0000,0,no promotion,51.900000


In [ ]:
# initialize flag to 0
df2['in_campaign'] = 0

# for each campaign interval in df1, mark the matching df2 rows
for _, cam in df1.iterrows():
    mask = (
        (df2['competitor'] == cam['competitor']) &
        (df2['time_key']    >= cam['start_date']) &
        (df2['time_key']    <= cam['end_date'])
    )
    df2.loc[mask, 'in_campaign'] = 1

df2.head()

,sku,time_key,competitor,pvp_was,discount,flag_promo,leaflet,effective_price,in_campaign
0,2056,2024-03-12,chain,21.70,0.0000,0,no promotion,21.700000,1
1,4435,2023-06-21,chain,18.92,0.2754,1,weekly,13.709432,0
2,1951,2023-03-03,competitorA,60.58,0.3524,1,no promotion,39.231608,1
3,2135,2024-06-05,chain,55.37,0.2509,1,weekly,41.477667,0
4,3949,2023-05-29,chain,51.90,0.0000,0,no promotion,51.900000,1


In [7]:
print(df2['in_campaign'].value_counts())

in_campaign
0    1823416
1    1288684
Name: count, dtype: int64


In [8]:
duplicates = df2.duplicated().sum()
print(f"Exact duplicate rows: {duplicates}")

Exact duplicate rows: 0


In [ ]:
# dataset 3
df3 = pd.read_csv("train/product_structures_sales.csv")

df3['time_key'] = pd.to_datetime(df3['time_key'].astype(str), format='%Y%m%d') #convert time_key to datetime
#df3 = df3.set_index(['time_key','sku']).sort_index()

#drop negative quantity values
neg_count = (df3['quantity'] < 0).sum()
print(f"Dropping {neg_count} rows with negative quantity")

# Filter them out
df3 = df3[df3['quantity'] >= 0].copy()

#transform structure_levels into categorical variables
structure_cols = [
    'structure_level_1',
    'structure_level_2',
    'structure_level_3',
    'structure_level_4'
]

for col in structure_cols:
    df3[col] = df3[col].astype('category')

df3.head()

Dropping 292 rows with negative quantity


,structure_level_4,structure_level_3,structure_level_2,structure_level_1,sku,time_key,quantity
0,3020206,30202,302,3,3111,2023-06-18,18.6840
1,3020608,30206,302,3,3278,2024-07-31,396.1008
2,3020809,30208,302,3,3603,2023-08-07,6.2280
3,3020608,30206,302,3,4604,2023-01-31,27.4032
4,3040808,30408,304,3,3041,2023-09-06,6.2280


In [10]:
df3.head()

,structure_level_4,structure_level_3,structure_level_2,structure_level_1,sku,time_key,quantity
0,3020206,30202,302,3,3111,2023-06-18,18.6840
1,3020608,30206,302,3,3278,2024-07-31,396.1008
2,3020809,30208,302,3,3603,2023-08-07,6.2280
3,3020608,30206,302,3,4604,2023-01-31,27.4032
4,3040808,30408,304,3,3041,2023-09-06,6.2280


In [ ]:
# Only get the competitor A and chain prices (eliminate competitor B from df2)
df2_comp_A = df2[df2['competitor'] != 'competitorB'].copy()

In [12]:
df_merged = pd.merge(
    df2_comp_A,
    df3,
    on=['sku','time_key'],
    how='left' #keep all df2 skus
)

df_merged.head()

,sku,time_key,competitor,pvp_was,discount,flag_promo,leaflet,effective_price,in_campaign,structure_level_4,structure_level_3,structure_level_2,structure_level_1,quantity
0,2056,2024-03-12,chain,21.70,0.0000,0,no promotion,21.700000,1,3021101,30211,302,3,146.980800
1,4435,2023-06-21,chain,18.92,0.2754,1,weekly,13.709432,0,3030708,30307,303,3,198.050400
2,1951,2023-03-03,competitorA,60.58,0.3524,1,no promotion,39.231608,1,3020806,30208,302,3,37.368000
3,2135,2024-06-05,chain,55.37,0.2509,1,weekly,41.477667,0,3010801,30108,301,3,229.190400
4,3949,2023-05-29,chain,51.90,0.0000,0,no promotion,51.900000,1,1010407,10104,101,1,4734.052272


In [13]:
duplicates_merged = df_merged.duplicated().sum()
print(f"Exact duplicate rows: {duplicates_merged}")

Exact duplicate rows: 0


In [14]:
df2['competitor'].value_counts()

competitor
chain          1749087
competitorA    1094290
competitorB     268723
Name: count, dtype: int64

In [ ]:
### separate data chain vs competitor A

# Split out the two series
df_chain = df_merged[df_merged['competitor'] == 'chain'].copy()
df_A = df_merged[df_merged['competitor'] == 'competitorA'].copy()

# Drop the old competitor column and rename the remaining columns
df_chain = df_chain.drop(columns='competitor').rename(columns={
    'pvp_was':      'pvp_was_chain',
    'discount':     'discount_chain',
    'quantity':   'quantity_chain',
    'flag_promo':   'flag_promo_chain',
    'leaflet':   'leaflet_chain',
    'effective_price': 'effective_price_chain'  # chain doesn't have effective_price target
})

df_A = df_A.drop(columns='competitor').rename(columns={
    'pvp_was':        'pvp_was_A',
    'discount':       'discount_A',
    'quantity':   'quantity_A',
    'flag_promo':   'flag_promo_A',
    'leaflet':   'leaflet_A',
    'effective_price': 'effective_price_A'
})

# Merge back together on sku & time_key
df_wide_prices = pd.merge(
    df_chain,
    df_A,
    on=['sku','time_key'],
    how='inner', 
)

In [17]:
df_wide_prices.head()

,sku,time_key,pvp_was_chain,discount_chain,flag_promo_chain,leaflet_chain,effective_price_chain,in_campaign_x,structure_level_4_x,structure_level_3_x,...,discount_A,flag_promo_A,leaflet_A,effective_price_A,in_campaign_y,structure_level_4_y,structure_level_3_y,structure_level_2_y,structure_level_1_y,quantity_A
0,4435,2023-06-21,18.92,0.2754,1,weekly,13.709432,0,3030708,30307,...,0.2754,1,weekly,13.709432,0,3030708,30307,303,3,198.050400
1,2135,2024-06-05,55.37,0.2509,1,weekly,41.477667,0,3010801,30108,...,0.0000,0,weekly,41.480000,0,3010801,30108,301,3,229.190400
2,3949,2023-05-29,51.90,0.0000,0,no promotion,51.900000,1,1010407,10104,...,0.0000,0,no promotion,51.900000,0,1010407,10104,101,1,4734.052272
3,2937,2023-04-05,17.18,0.0000,0,no promotion,17.180000,1,3020601,30206,...,0.0000,0,no promotion,17.180000,0,3020601,30206,302,3,280.260000
4,4019,2024-07-10,34.54,0.0000,0,no promotion,34.540000,0,2020507,20205,...,0.0000,0,no promotion,34.370000,0,2020507,20205,202,2,44.841600


In [18]:
df = df_wide_prices.drop(columns=['effective_price_chain', 'pvp_was_A', 'discount_A', 'quantity_A', 'structure_level_4_y', 'structure_level_3_y', 'structure_level_2_y', 'structure_level_1_y'])

In [19]:
df = df.rename(columns={
    'structure_level_4_x': 'structure_level_4',
    'structure_level_3_x': 'structure_level_3',
    'structure_level_2_x': 'structure_level_2',
    'structure_level_1_x': 'structure_level_1'
})


In [20]:
df.head()

,sku,time_key,pvp_was_chain,discount_chain,flag_promo_chain,leaflet_chain,in_campaign_x,structure_level_4,structure_level_3,structure_level_2,structure_level_1,quantity_chain,flag_promo_A,leaflet_A,effective_price_A,in_campaign_y
0,4435,2023-06-21,18.92,0.2754,1,weekly,0,3030708,30307,303,3,198.050400,1,weekly,13.709432,0
1,2135,2024-06-05,55.37,0.2509,1,weekly,0,3010801,30108,301,3,229.190400,0,weekly,41.480000,0
2,3949,2023-05-29,51.90,0.0000,0,no promotion,1,1010407,10104,101,1,4734.052272,0,no promotion,51.900000,0
3,2937,2023-04-05,17.18,0.0000,0,no promotion,1,3020601,30206,302,3,280.260000,0,no promotion,17.180000,0
4,4019,2024-07-10,34.54,0.0000,0,no promotion,0,2020507,20205,202,2,44.841600,0,no promotion,34.370000,0


In [21]:
df['month'] = df['time_key'].dt.month        
df['day_of_week'] = df['time_key'].dt.weekday      
df['day_of_month']= df['time_key'].dt.day         
df['year'] = df['time_key'].dt.year
df['is_weekend']  = (df['time_key'].dt.weekday >= 5).astype(int)

In [ ]:
### Feature Engineering Lags

cols_to_lag = ['pvp_was_chain', 'discount_chain']

# define the lag days
lag_days = [1, 7, 14, 30]

for col in cols_to_lag:
    for lag in lag_days:
        df[f'{col}_lag{lag}'] = (
            df
              .groupby('sku')[col]
              .shift(lag)
        )

df.head()

,sku,time_key,pvp_was_chain,discount_chain,flag_promo_chain,leaflet_chain,in_campaign_x,structure_level_4,structure_level_3,structure_level_2,...,year,is_weekend,pvp_was_chain_lag1,pvp_was_chain_lag7,pvp_was_chain_lag14,pvp_was_chain_lag30,discount_chain_lag1,discount_chain_lag7,discount_chain_lag14,discount_chain_lag30
0,4435,2023-06-21,18.92,0.2754,1,weekly,0,3030708,30307,303,...,2023,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2135,2024-06-05,55.37,0.2509,1,weekly,0,3010801,30108,301,...,2024,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3949,2023-05-29,51.90,0.0000,0,no promotion,1,1010407,10104,101,...,2023,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2937,2023-04-05,17.18,0.0000,0,no promotion,1,3020601,30206,302,...,2023,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4019,2024-07-10,34.54,0.0000,0,no promotion,0,2020507,20205,202,...,2024,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
### Feature Engineering Rolling Window

cols_to_roll = ['pvp_was_chain', 'discount_chain']

# define window sizes
windows = [7, 14, 30]

# compute rolling means and rolling stds
for col in cols_to_roll:
    for w in windows:
        df[f'{col}_roll{w}_mean'] = (
            df.groupby('sku')[col]
              .transform(lambda s: s.rolling(window=w, min_periods=1).mean())
        )
        df[f'{col}_roll{w}_std'] = (
            df.groupby('sku')[col]
              .transform(lambda s: s.rolling(window=w, min_periods=1).std().fillna(0))
        )
        
df.head()

,sku,time_key,pvp_was_chain,discount_chain,flag_promo_chain,leaflet_chain,in_campaign_x,structure_level_4,structure_level_3,structure_level_2,...,pvp_was_chain_roll14_mean,pvp_was_chain_roll14_std,pvp_was_chain_roll30_mean,pvp_was_chain_roll30_std,discount_chain_roll7_mean,discount_chain_roll7_std,discount_chain_roll14_mean,discount_chain_roll14_std,discount_chain_roll30_mean,discount_chain_roll30_std
0,4435,2023-06-21,18.92,0.2754,1,weekly,0,3030708,30307,303,...,18.92,0.0,18.92,0.0,0.2754,0.0,0.2754,0.0,0.2754,0.0
1,2135,2024-06-05,55.37,0.2509,1,weekly,0,3010801,30108,301,...,55.37,0.0,55.37,0.0,0.2509,0.0,0.2509,0.0,0.2509,0.0
2,3949,2023-05-29,51.90,0.0000,0,no promotion,1,1010407,10104,101,...,51.90,0.0,51.90,0.0,0.0000,0.0,0.0000,0.0,0.0000,0.0
3,2937,2023-04-05,17.18,0.0000,0,no promotion,1,3020601,30206,302,...,17.18,0.0,17.18,0.0,0.0000,0.0,0.0000,0.0,0.0000,0.0
4,4019,2024-07-10,34.54,0.0000,0,no promotion,0,2020507,20205,202,...,34.54,0.0,34.54,0.0,0.0000,0.0,0.0000,0.0,0.0000,0.0


In [ ]:
# List all lag columns
lag_cols = [f'{col}_lag{lag}'
            for col in cols_to_lag
            for lag in lag_days]

# Drop rows with any NaN in those
df = df.dropna(subset=lag_cols, how='any')\
       .reset_index(drop=True)
       
df.head()

,sku,time_key,pvp_was_chain,discount_chain,flag_promo_chain,leaflet_chain,in_campaign_x,structure_level_4,structure_level_3,structure_level_2,...,pvp_was_chain_roll14_mean,pvp_was_chain_roll14_std,pvp_was_chain_roll30_mean,pvp_was_chain_roll30_std,discount_chain_roll7_mean,discount_chain_roll7_std,discount_chain_roll14_mean,discount_chain_roll14_std,discount_chain_roll30_mean,discount_chain_roll30_std
0,3924,2024-04-02,25.17,0.0000,0,no promotion,1,2011101,20111,201,...,31.415000,6.939765,30.958000,6.354245,0.059986,0.079652,0.066107,0.074801,0.060597,0.069158
1,3924,2024-08-26,25.17,0.0691,1,weekly,0,2011101,20111,201,...,31.415000,6.939765,30.917667,6.388055,0.057671,0.079027,0.071043,0.072343,0.060057,0.069022
2,4251,2023-07-08,142.15,0.3101,1,themed,1,2010807,20108,201,...,198.907857,47.534495,200.267667,48.390859,0.339986,0.050630,0.328479,0.060712,0.322327,0.062895
3,4251,2023-07-08,142.15,0.3101,1,themed,1,2010807,20108,201,...,193.527143,49.493642,196.622667,48.516941,0.339900,0.050689,0.332729,0.056797,0.318860,0.060482
4,4251,2023-11-28,217.48,0.2506,1,themed,0,2010807,20108,201,...,196.056429,49.767134,197.803000,48.581380,0.316543,0.048418,0.332243,0.057515,0.318633,0.060733


In [25]:
df = df.sort_values('time_key').reset_index(drop=True) #sort by time_key

In [26]:
df.head()

,sku,time_key,pvp_was_chain,discount_chain,flag_promo_chain,leaflet_chain,in_campaign_x,structure_level_4,structure_level_3,structure_level_2,...,pvp_was_chain_roll14_mean,pvp_was_chain_roll14_std,pvp_was_chain_roll30_mean,pvp_was_chain_roll30_std,discount_chain_roll7_mean,discount_chain_roll7_std,discount_chain_roll14_mean,discount_chain_roll14_std,discount_chain_roll30_mean,discount_chain_roll30_std
0,3290,2023-01-03,42.52,0.0,0,no promotion,1,2020516,20205,202,...,42.520000,0.000000,42.520000,0.000000,0.036157,0.095663,0.062393,0.124656,0.089253,0.139297
1,2355,2023-01-03,18.92,0.0,0,no promotion,1,3031106,30311,303,...,18.273571,0.598416,18.224000,0.572747,0.045829,0.043938,0.048321,0.038682,0.041683,0.038106
2,3226,2023-01-03,69.25,0.0,0,no promotion,1,3030812,30308,303,...,85.370000,4.639655,86.031333,3.169488,0.015729,0.041614,0.007864,0.029425,0.011010,0.033595
3,3983,2023-01-03,34.54,0.0,0,no promotion,1,3030814,30308,303,...,39.005714,1.891941,39.229000,1.589720,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,1696,2023-01-03,17.18,0.0,0,no promotion,1,3020313,30203,302,...,17.180000,0.000000,17.180000,0.000000,0.000000,0.000000,0.018043,0.067510,0.018510,0.070749


In [27]:
feature_cols = [
    'sku', 'pvp_was_chain', 'discount_chain', 'structure_level_3', 'structure_level_2', 'structure_level_1',
    'structure_level_4', 'quantity_chain', 'month', 'day_of_week', 'day_of_month', 'year', 'flag_promo_chain', 'flag_promo_A', 'leaflet_chain', 'leaflet_A', 'pvp_was_chain_lag1', 'pvp_was_chain_lag7', 'pvp_was_chain_lag14', 'pvp_was_chain_lag30', 'discount_chain_lag1', 'discount_chain_lag7', 'discount_chain_lag14', 'discount_chain_lag30']

X = df[feature_cols]
y = df['effective_price_A']

In [28]:
### train test split time-based & split into X and y (train and test sets)
split = int(len(df) * 0.8)

train_X, test_X = X.iloc[:split], X.iloc[split:]
train_y, test_y = y.iloc[:split], y.iloc[split:]

print(f"Train from {df['time_key'].iloc[0].date()} to {df['time_key'].iloc[split-1].date()}")
print(f"Test  from {df['time_key'].iloc[split].date()} to {df['time_key'].iloc[-1].date()}")

Train from 2023-01-03 to 2024-06-15
Test  from 2024-06-15 to 2024-10-28


In [29]:
categorical_cols = ['sku','structure_level_4', 'month', 'day_of_week', 'flag_promo_chain', 'flag_promo_A', 'leaflet_chain', 'leaflet_A', 'structure_level_3', 'structure_level_2', 'structure_level_1']
numeric_cols     = ['pvp_was_chain', 'discount_chain', 'quantity_chain', 'year', 'day_of_month', 'pvp_was_chain_lag1', 'pvp_was_chain_lag7', 'pvp_was_chain_lag14', 'pvp_was_chain_lag30', 'discount_chain_lag1', 'discount_chain_lag7', 'discount_chain_lag14', 'discount_chain_lag30']

In [30]:
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ("num", RobustScaler(), numeric_cols)
])

In [31]:
### Model pipeline
def model_pipeline(train_X, test_X, train_y, test_y, random_state=42):

    clf = Pipeline([
        ('preproc',preprocessor),
        ('regressor', LGBMRegressor(n_estimators=100, random_state=random_state, n_jobs=-1))])

    clf.fit(train_X, train_y)

    y_pred = clf.predict(test_X)
    
    mae = mean_absolute_error(test_y, y_pred)
    print(f"MAE: {mae}")
    
    mape = (abs(test_y - y_pred) / test_y).mean() * 100
    print(f"MAPE: {mape:.1f}%")

    return clf, y_pred, mae, mape

In [32]:
### Model evaluation (MAE)
model, preds, mae, mape = model_pipeline(train_X, test_X, train_y, test_y)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.106456 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7929
[LightGBM] [Info] Number of data points in the train set: 857828, number of used features: 2573
[LightGBM] [Info] Start training from score 48.047363
MAE: 2.1192234727191903
MAPE: 4.9%
